# Fink/LSST — HEALPix Skymap Statistics

Ce notebook produit des skymaps HEALPix des alertes Fink/LSST :
- **Statistique totale** : densité de toutes les alertes sur le ciel
- **Statistique par tag** : `extragalactic_lt20mag_candidate`, `extragalactic_new_candidate`, `hostless_candidate`, `in_tns`, `sn_near_galaxy_candidate`, + objets système solaire (SSO)

Annotations : plan galactique, centre galactique, Nuages de Magellan, Deep Drilling Fields Rubin/LSST.

---
**Dépendances** : `healpy`, `astropy`, `numpy`, `matplotlib`, `requests`, `pandas`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import healpy as hp
import requests
from astropy.coordinates import SkyCoord, Galactic
import astropy.units as u
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print(f'healpy version : {hp.__version__}')
print(f'numpy  version : {np.__version__}')

## 1. Configuration

In [ ]:
# ---- Paramètres HEALPix ----
NSIDE = 64        # résolution : ~55 arcmin par pixel (NSIDE=128 → ~27 arcmin, plus lent)
NPIX  = hp.nside2npix(NSIDE)
print(f'NSIDE={NSIDE}  →  {NPIX} pixels, résolution ~{hp.nside2resol(NSIDE, arcmin=True):.1f} arcmin')

# ---- API Fink LSST ----
FINK_API = 'https://api.lsst.fink-portal.org'

# ---- Tags disponibles ----
TAGS = [
    'extragalactic_lt20mag_candidate',
    'extragalactic_new_candidate',
    'hostless_candidate',
    'in_tns',
    'sn_near_galaxy_candidate',
]

# Nombre max d'alertes par tag (augmenter pour statistiques plus complètes)
N_ALERTS_PER_TAG = 10000
N_SSO            = 10000   # Objets système solaire

## 2. Annotations astronomiques

In [ ]:
def galactic_plane_radec(n_points=500):
    """Retourne RA/Dec du plan galactique (b=0)."""
    l_range = np.linspace(0, 360, n_points) * u.deg
    coords  = SkyCoord(l=l_range, b=np.zeros(n_points)*u.deg, frame='galactic')
    icrs    = coords.icrs
    return icrs.ra.deg, icrs.dec.deg

def galactic_band_radec(b_deg=10, n_points=500):
    """Retourne RA/Dec d'une latitude galactique b_deg."""
    l_range = np.linspace(0, 360, n_points) * u.deg
    coords  = SkyCoord(l=l_range, b=np.full(n_points, b_deg)*u.deg, frame='galactic')
    icrs    = coords.icrs
    return icrs.ra.deg, icrs.dec.deg

# ---- Points remarquables ----
LANDMARKS = {
    'Centre galactique': SkyCoord(l=0*u.deg, b=0*u.deg, frame='galactic').icrs,
    'LMC':               SkyCoord(ra=80.89*u.deg,  dec=-69.76*u.deg,  frame='icrs'),
    'SMC':               SkyCoord(ra=13.16*u.deg,  dec=-72.80*u.deg,  frame='icrs'),
}

# ---- Deep Drilling Fields Rubin/LSST (v3.3) ----
# Source : https://www.lsst.org/scientists/survey-design
DDF = {
    'COSMOS':      SkyCoord(ra=150.1191*u.deg, dec=2.2058*u.deg,   frame='icrs'),
    'XMM-LSS':     SkyCoord(ra=34.3900*u.deg,  dec=-4.9000*u.deg,  frame='icrs'),
    'ECDFS':       SkyCoord(ra=53.1250*u.deg,  dec=-28.1000*u.deg, frame='icrs'),
    'EDFS-a':      SkyCoord(ra=58.9000*u.deg,  dec=-49.3150*u.deg, frame='icrs'),
    'EDFS-b':      SkyCoord(ra=63.6000*u.deg,  dec=-47.6000*u.deg, frame='icrs'),
    'Euclid Deep Field South': SkyCoord(ra=61.2400*u.deg, dec=-48.4230*u.deg, frame='icrs'),
}

print('Annotations définies :')
for name, coord in {**LANDMARKS, **DDF}.items():
    print(f'  {name:35s}  RA={coord.ra.deg:.2f}°  Dec={coord.dec.deg:.2f}°')

## 3. Fonctions utilitaires

In [ ]:
def radec_to_healpix(ra_deg, dec_deg, nside=NSIDE):
    """Convertit RA/Dec (degrés) → indices pixels HEALPix (RING ordering)."""
    theta = np.radians(90.0 - dec_deg)   # colatitude
    phi   = np.radians(ra_deg)
    return hp.ang2pix(nside, theta, phi)

def fetch_tag_alerts(tag, n=N_ALERTS_PER_TAG, columns='r:ra,r:dec,r:midpointMjdTai'):
    """Récupère les alertes d'un tag Fink via l'API REST."""
    url = f'{FINK_API}/api/v1/tags'
    params = {'tag': tag, 'n': n, 'columns': columns, 'output-format': 'csv'}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text))
    # Normaliser les noms de colonnes (enlever préfixe 'r:')
    df.columns = [c.replace('r:', '') for c in df.columns]
    print(f'  [{tag:40s}]  {len(df):6d} alertes')
    return df

def fetch_sso_alerts(n=N_SSO):
    """Récupère les alertes SSO (Système Solaire) depuis Fink."""
    url = f'{FINK_API}/api/v1/sso'
    # L'endpoint SSO retourne par objet — on fait une conesearch large
    # Alternative : utiliser l'endpoint /api/v1/objects avec filtre SSO
    # Pour l'instant on utilise le tag via l'API sources générales
    # Note : les SSO ont un endpoint dédié mais nécessitent un ssnamenr
    # On interroge via latests avec classe 'Solar System candidate'
    url2 = f'{FINK_API}/api/v1/latests'
    params = {
        'class': 'Solar System candidate',
        'n': n,
        'columns': 'i:ra,i:dec,i:jd',
        'output-format': 'csv'
    }
    try:
        r = requests.get(url2, params=params, timeout=120)
        r.raise_for_status()
        df = pd.read_csv(StringIO(r.text))
        df.columns = [c.replace('i:', '') for c in df.columns]
        # Renommer jd → midpointMjdTai (approx)
        if 'jd' in df.columns:
            df = df.rename(columns={'jd': 'midpointMjdTai'})
        print(f'  [{"Solar System candidate":40s}]  {len(df):6d} alertes')
        return df
    except Exception as e:
        print(f'  SSO fetch failed : {e}')
        return pd.DataFrame(columns=['ra', 'dec', 'midpointMjdTai'])

def make_skymap(ra_deg, dec_deg, nside=NSIDE, label=''):
    """Construit une skymap HEALPix (count par pixel) à partir de RA/Dec."""
    skymap = np.zeros(hp.nside2npix(nside))
    mask   = np.isfinite(ra_deg) & np.isfinite(dec_deg)
    pix    = radec_to_healpix(ra_deg[mask], dec_deg[mask], nside)
    np.add.at(skymap, pix, 1)
    nonzero = np.sum(skymap > 0)
    print(f'  Skymap {label:40s}: {int(skymap.sum()):7d} entrées, {nonzero:5d} pixels occupés')
    return skymap

print('Fonctions utilitaires définies.')

## 4. Récupération des données

In [ ]:
print('=== Récupération des alertes par tag ===')
dfs = {}
for tag in TAGS:
    try:
        dfs[tag] = fetch_tag_alerts(tag)
    except Exception as e:
        print(f'  ERREUR {tag}: {e}')
        dfs[tag] = pd.DataFrame(columns=['ra', 'dec', 'midpointMjdTai'])

print('\n=== Récupération des alertes SSO ===')
dfs['Solar System candidate'] = fetch_sso_alerts()

print(f'\n=== Résumé ===' )
total_all = 0
for tag, df in dfs.items():
    print(f'  {tag:45s} : {len(df):6d} alertes')
    total_all += len(df)
print(f'  {"TOTAL":45s} : {total_all:6d} alertes')

## 5. Construction des skymaps HEALPix

In [ ]:
print('=== Construction des skymaps HEALPix ===')
skymaps = {}

# Skymap totale (union de tous les tags)
all_ra  = np.concatenate([df['ra'].values  for df in dfs.values() if len(df) > 0])
all_dec = np.concatenate([df['dec'].values for df in dfs.values() if len(df) > 0])
skymaps['Total'] = make_skymap(all_ra, all_dec, label='Total')

# Skymaps par tag
for tag, df in dfs.items():
    if len(df) > 0:
        skymaps[tag] = make_skymap(df['ra'].values, df['dec'].values, label=tag)
    else:
        skymaps[tag] = np.zeros(hp.nside2npix(NSIDE))

print('\nSkymaps construites.')

## 6. Fonction de visualisation avec annotations

In [ ]:
TAG_COLORS = {
    'extragalactic_lt20mag_candidate': 'plasma',
    'extragalactic_new_candidate':     'inferno',
    'hostless_candidate':              'magma',
    'in_tns':                          'viridis',
    'sn_near_galaxy_candidate':        'cividis',
    'Solar System candidate':          'YlOrRd',
    'Total':                           'hot',
}

def plot_skymap_with_annotations(
    skymap, title='', cmap='hot', unit='Nombre d\'alertes',
    log_scale=True, figsize=(14, 7), coord='C',
    show_galactic_plane=True, show_landmarks=True, show_ddf=True,
    ax=None, show=True
):
    """
    Visualise une skymap HEALPix en projection Mollweide avec annotations.

    Parameters
    ----------
    skymap : np.ndarray
        Carte HEALPix (RING ordering).
    log_scale : bool
        Si True, affiche log10(1 + count).
    coord : str
        'C' = équatorial, 'G' = galactique.
    """
    map_to_plot = np.log10(1 + skymap) if log_scale else skymap.copy()
    map_to_plot[map_to_plot == 0] = hp.UNSEEN

    plt.figure(figsize=figsize)
    hp.mollview(
        map_to_plot,
        coord=['C', coord] if coord != 'C' else ['C'],
        cmap=cmap,
        title=title,
        unit=f'log10(1+N)' if log_scale else unit,
        hold=False,
        margins=(0.01, 0.04, 0.01, 0.04),
    )
    hp.graticule(dpar=30, dmer=60, alpha=0.3, color='white', lw=0.5)

    # ---- Plan galactique ----
    if show_galactic_plane:
        for b_val, ls, lw, alpha in [(0, '-', 1.8, 0.9), (10, '--', 0.8, 0.5), (-10, '--', 0.8, 0.5)]:
            ra_gp, dec_gp = galactic_band_radec(b_val)
            # Trier par RA pour éviter les artefacts de connexion
            idx = np.argsort(ra_gp)
            # Couper aux discontinuités (saut > 180°)
            ra_s, dec_s = ra_gp[idx], dec_gp[idx]
            jumps = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
            segs_ra  = np.split(ra_s,  jumps)
            segs_dec = np.split(dec_s, jumps)
            for seg_ra, seg_dec in zip(segs_ra, segs_dec):
                hp.projplot(
                    np.radians(90 - seg_dec), np.radians(seg_ra),
                    linestyle=ls, linewidth=lw, alpha=alpha,
                    color='cyan', lonlat=False
                )

    # ---- Landmarks ----
    if show_landmarks:
        landmark_styles = {
            'Centre galactique': dict(marker='*', color='yellow', s=150, zorder=5),
            'LMC':               dict(marker='o', color='lime',   s=80,  zorder=5),
            'SMC':               dict(marker='o', color='lime',   s=60,  zorder=5),
        }
        for name, coord_obj in LANDMARKS.items():
            ra_r  = np.radians(coord_obj.ra.deg)
            dec_r = np.radians(90 - coord_obj.dec.deg)
            style = landmark_styles.get(name, dict(marker='x', color='white', s=60))
            hp.projscatter(dec_r, ra_r, lonlat=False, **style)
            hp.projtext(
                dec_r, ra_r,
                f'  {name}', color=style['color'],
                fontsize=7, fontweight='bold', lonlat=False
            )

    # ---- Deep Drilling Fields ----
    if show_ddf:
        for name, coord_obj in DDF.items():
            ra_r  = np.radians(coord_obj.ra.deg)
            dec_r = np.radians(90 - coord_obj.dec.deg)
            hp.projscatter(dec_r, ra_r, lonlat=False, marker='D',
                           color='white', s=50, edgecolors='orange',
                           linewidths=1.5, zorder=6)
            hp.projtext(
                dec_r, ra_r, f'  {name}',
                color='orange', fontsize=6.5, fontstyle='italic', lonlat=False
            )

    # ---- Légende ----
    handles = [
        mpatches.Patch(color='cyan',   label='Plan galactique (|b|≤10°)'),
        plt.Line2D([0],[0], color='yellow', marker='*', ms=10,
                   ls='None', label='Centre galactique'),
        plt.Line2D([0],[0], color='lime',   marker='o', ms=7,
                   ls='None', label='Nuages de Magellan'),
        plt.Line2D([0],[0], color='orange', marker='D', ms=7,
                   ls='None', markerfacecolor='white',
                   markeredgecolor='orange', label='Deep Drilling Fields'),
    ]
    plt.gca().legend(handles=handles, loc='lower right', fontsize=7,
                     framealpha=0.7, facecolor='#1a1a2e')

    n_total = int(skymap[skymap > 0].sum())
    plt.figtext(0.12, 0.02, f'N alertes = {n_total:,}  |  NSIDE={NSIDE}',
                fontsize=8, color='gray')
    if show:
        plt.tight_layout()
        plt.show()

print('Fonction de visualisation définie.')

## 7. Skymap totale

In [ ]:
plot_skymap_with_annotations(
    skymaps['Total'],
    title='Fink/LSST — Distribution spatiale totale des alertes (projection équatoriale)',
    cmap='hot',
    log_scale=True,
)

In [ ]:
# Même carte en coordonnées galactiques
plot_skymap_with_annotations(
    skymaps['Total'],
    title='Fink/LSST — Distribution spatiale totale (projection galactique)',
    cmap='hot',
    log_scale=True,
    coord='G',
)

## 8. Skymap par tag

In [ ]:
all_tags_ordered = TAGS + ['Solar System candidate']

for tag in all_tags_ordered:
    sm = skymaps.get(tag, np.zeros(NPIX))
    n  = int(sm.sum())
    if n == 0:
        print(f'Pas de données pour : {tag}')
        continue
    cmap = TAG_COLORS.get(tag, 'viridis')
    plot_skymap_with_annotations(
        sm,
        title=f'Fink/LSST — {tag}',
        cmap=cmap,
        log_scale=True,
    )

## 9. Panneau multi-tags comparatif

In [ ]:
valid_tags = [t for t in all_tags_ordered if skymaps.get(t, np.zeros(1)).sum() > 0]
n_tags     = len(valid_tags)
ncols      = 2
nrows      = int(np.ceil(n_tags / ncols))

fig, _ = plt.subplots(nrows, ncols, figsize=(20, 6 * nrows))
plt.subplots_adjust(hspace=0.1, wspace=0.01)


for i, tag in enumerate(valid_tags):
    plt.subplot(nrows, ncols, i + 1)
    sm = skymaps[tag]
    map_to_plot = np.log10(1 + sm)
    map_to_plot[map_to_plot == 0] = hp.UNSEEN

    hp.mollview(
        map_to_plot,
        cmap=TAG_COLORS.get(tag, 'viridis'),
        title=f'{tag}\n(N={int(sm.sum()):,})',
        unit='log10(1+N)',
        hold=True,
        sub=(nrows, ncols, i + 1),
        margins=(0.01, 0.07, 0.01, 0.04),
    )
    hp.graticule(dpar=30, dmer=60, alpha=0.25, color='white', lw=0.4)

    # Plan galactique simplifié
    ra_gp, dec_gp = galactic_plane_radec(200)
    idx = np.argsort(ra_gp)
    ra_s, dec_s = ra_gp[idx], dec_gp[idx]
    jumps = np.where(np.abs(np.diff(ra_s)) > 180)[0] + 1
    for seg_ra, seg_dec in zip(np.split(ra_s, jumps), np.split(dec_s, jumps)):
        hp.projplot(np.radians(90 - seg_dec), np.radians(seg_ra),
                    '-', linewidth=1.0, alpha=0.7, color='cyan', lonlat=False)

    # DDFs
    for name, coord_obj in DDF.items():
        hp.projscatter(
            np.radians(90 - coord_obj.dec.deg), np.radians(coord_obj.ra.deg),
            marker='D', color='white', s=25,
            edgecolors='orange', linewidths=1.2, zorder=6, lonlat=False
        )

# Masquer les sous-graphes vides
for j in range(n_tags, nrows * ncols):
    plt.subplot(nrows, ncols, j + 1)
    plt.axis('off')

plt.suptitle('Fink/LSST — Skymaps HEALPix par tag de classification',
             fontsize=14, y=1.01, fontweight='bold')
plt.savefig('fink_healpix_multitag.pdf', bbox_inches='tight', dpi=150)
plt.savefig('fink_healpix_multitag.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figures sauvegardées : fink_healpix_multitag.pdf / .png')

## 10. Statistiques de recouvrement entre tags

In [ ]:
# Matrice de corrélation spatiale entre tags (pixels communs occupés)
from itertools import combinations

print('=== Pixels occupés par tag ===')
occupied = {}
for tag in valid_tags:
    pix_set = set(np.where(skymaps[tag] > 0)[0])
    occupied[tag] = pix_set
    frac = len(pix_set) / NPIX * 100
    print(f'  {tag:45s}: {len(pix_set):5d} pixels  ({frac:.2f}% du ciel)')

print('\n=== Recouvrement entre paires de tags (fraction de pixels communs) ===')
for t1, t2 in combinations(valid_tags, 2):
    inter = len(occupied[t1] & occupied[t2])
    union = len(occupied[t1] | occupied[t2])
    jaccard = inter / union if union > 0 else 0
    print(f'  {t1[:25]:25s} ∩ {t2[:25]:25s} : IoU={jaccard:.3f}  ({inter} pixels communs)')

## 11. Densité en coordonnées galactiques (vérification biais)

In [ ]:
# Distribution en latitude galactique pour tous les tags
fig, axes = plt.subplots(len(valid_tags), 1, figsize=(12, 3 * len(valid_tags)), sharex=True)
if len(valid_tags) == 1:
    axes = [axes]

b_bins = np.linspace(-90, 90, 73)  # bins de 2.5°
b_mid  = 0.5 * (b_bins[:-1] + b_bins[1:])

for ax, tag in zip(axes, valid_tags):
    df = dfs[tag]
    if len(df) == 0:
        ax.set_title(f'{tag} — no data')
        continue
    coords = SkyCoord(ra=df['ra'].values*u.deg, dec=df['dec'].values*u.deg, frame='icrs')
    gal    = coords.galactic
    b_vals = gal.b.deg
    counts, _ = np.histogram(b_vals, bins=b_bins)
    # Normaliser par l'aire du bin (correction cosinus)
    cos_corr = np.cos(np.radians(b_mid))
    density  = counts / (cos_corr + 1e-10)
    ax.bar(b_mid, density, width=2.5, color=plt.get_cmap(TAG_COLORS.get(tag, 'viridis'))(0.6), alpha=0.8)
    ax.axvline(0, color='cyan', ls='--', lw=1, alpha=0.7, label='Plan galactique')
    ax.set_ylabel('Densité\n(corr. cos b)', fontsize=8)
    ax.set_title(f'{tag}  (N={len(df):,})', fontsize=9)
    ax.legend(fontsize=7)

axes[-1].set_xlabel('Latitude galactique b (°)', fontsize=10)
plt.suptitle('Distribution en latitude galactique par tag', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fink_latitude_galactique_par_tag.pdf', bbox_inches='tight', dpi=150)
plt.show()